In [ ]:
import sys

# Add the new path
new_path2 = "/home/michele/code/michele_mmdet3d/"
if not new_path2 in sys.path:
    sys.path.insert(1, new_path2)

# Print the version of MMCV
import mmcv
print(mmcv.__version__)
print(mmcv.__file__)

# Import the inferencer
from mmdet3d.apis import MultiModalityDet3DInferencer

In [ ]:
#################################################################################################################
#                                           Initialize Inferencer                                               #
#################################################################################################################



# Initialize the inferencer class

################################### STANDARD ###################################
inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/configs/minerva/MINERVA_mvxnet.py",
                                          weights="/home/michele/code/michele_mmdet3d/work_dirs/MINERVA_mvxnet/epoch_180.pth",
                                          show_progress=False)

################################### MVX-NET SMALL VOXEL ###################################
# inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/configs/minerva/MINERVA_mvxnet.py",
#                                           weights="/home/michele/code/michele_mmdet3d/work_dirs/MINERVA_mvxnet/epoch_150_small_voxels.pth",
#                                           show_progress=False)

################################### MVX-NET SMALL VOXEL | DEEP CONVOLUTION  ###################################
# inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/configs/minerva/MINERVA_mvxnet.py",
#                                           weigths="/home/michele/code/michele_mmdet3d/work_dirs/MINERVA_mvxnet/epoch_170_small_voxels_deep_conv.pth",
#                                           show_progress=False)



# Define a function that prints the results of the inference
def print_results(results):
    import numpy as np
    for element in results: 
        print("\n\n")
        print(f"scores:\t{element['predictions'][0]['scores_3d']}")
        bboxes = np.array(element['predictions'][0]['bboxes_3d']) 
        num_boxes = bboxes.shape[0]
        for i in range(min(num_boxes, 3)):
            print(f"x: {bboxes[i][0]:.0f}\ty: {bboxes[i][1]:.0f}\tz: {bboxes[i][2]:.0f}")

In [ ]:
#################################################################################################################
#                                               Just Inference                                                  #
#################################################################################################################



# Needed for the handling 
import copy

# Read the files in validation list
val_list_txt_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/ImageSets/val.txt"
with open(val_list_txt_file, 'r') as file:
    val_file_names = [line.strip() for line in file]

# Choose a smaller set of the validation files
one_every_n = 1
max_number = 1e3
wait_time_default = 2.5
pred_score_thr_default = 0.2
based_inputs = []
for i, name in enumerate(val_file_names):
    if i%one_every_n==0 and len(based_inputs)<max_number:
        based_inputs.append(dict(
            points=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/training/velodyne_reduced/"+name+".bin"),
            img=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/training/image_2/"+name+".png"),
            infos=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/minerva_polimove_infos_val.pkl")
        ))

# Create a deep copy of the inputs
# NOTE:
#   - Needed because the dictionaries are then modified dinamically
#   - Different from the LIDAR version
inputs = copy.deepcopy(based_inputs)

print(f"Total validation point_clouds: {len(val_file_names)}")
print(f"\tOne every n: {one_every_n}")
print(f"\tMax number: {max_number}")
print(f"\tSelected point_clouds: {len(inputs)}")

# Do the inference
results = []
for input in inputs:
    results.append(inferencer(input))

# Print the information about the predictions
print_results(results)

In [ ]:
#################################################################################################################
#                                           Inference and Visualize                                             #
#################################################################################################################



# Set the matplotlib library so that the correct window is visualized
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

# Again, need the deep copy
inputs = copy.deepcopy(based_inputs)

# Do the inference with visualization
for i, input in enumerate(inputs):
    print(f"\n____________________________\n{i+1} out of {len(inputs)}")
    inferencer(input, show=True, wait_time=wait_time_default, pred_score_thr=pred_score_thr_default)